# AutoGCMS Pipeline

Pipeline automatizado para procesamiento de datos GC-MS.

**Flujo por muestra:**
1. Lectura y preprocesado del archivo .cdf
2. Deteccion automatica de picos en el cromatograma (TIC)
3. Extraccion de espectros de masas en cada pico
4. Identificacion de compuestos contra libreria NIST exportada (.msp) usando similitud coseno

**Requisito:** Exportar MAINLIB desde NIST MS Search a formato .msp y configurar la ruta en `config.py` (`NIST_MSP_PATH`).

---

In [ ]:
# Setup
import sys
from pathlib import Path

# Detectar el directorio del proyecto
_candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_DIR = next((p for p in _candidates if (p / "config.py").exists()), _candidates[-1])

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

import config
from pipeline import setup_logging, list_cdf_files

setup_logging()
print(f"Directorio de datos: {config.DATA_DIR}")
print(f"Directorio de salida: {config.OUTPUT_DIR}")
print(f"Libreria .msp: {config.NIST_MSP_PATH}")
print(f"  Existe: {config.NIST_MSP_PATH.exists()}")

## 1. Seleccionar archivo .cdf

Cambia `FILE_IDX` para procesar otra muestra.

In [ ]:
cdf_files = list_cdf_files()
print(f"Archivos .cdf disponibles:\n")
for i, f in enumerate(cdf_files):
    print(f"  [{i}] {f.name}")

# === SELECCIONAR AQUI ===
FILE_IDX = 0
# ========================

test_file = cdf_files[FILE_IDX]
print(f"\nSeleccionado: {test_file.name}")

---
## 2. Preprocesado del cromatograma (Modulo 1)

Lee el archivo .cdf, aplica suavizado Savitzky-Golay y correccion de baseline.

In [ ]:
from preprocessing import preprocess

data = preprocess(test_file)
print(f"Muestra: {data.name}")
print(f"Scans: {data.n_scans}")
print(f"Tiempo: {data.scan_times_min[0]:.2f} - {data.scan_times_min[-1]:.2f} min")
print(f"Rango m/z: {data.mass_range[0]:.1f} - {data.mass_range[1]:.1f}")

In [ ]:
from visualization import plot_tic_preprocessing
fig = plot_tic_preprocessing(data)
plt.show()

---
## 3. Deteccion de picos (Modulo 2)

In [ ]:
from peak_detection import find_and_filter_peaks

peaks = find_and_filter_peaks(data)
print(f"Picos detectados: {peaks.n_peaks}")
print(f"\nTop 10 picos por intensidad:")
peaks.table.nlargest(10, 'intensity')[['peak_id', 'rt_min', 'intensity', 'snr', 'width_sec']]

In [ ]:
from visualization import plot_peaks, plot_peak_quality

fig = plot_peaks(data, peaks)
plt.show()

fig = plot_peak_quality(peaks)
plt.show()

---
## 4. Extraccion de espectros de masas (Modulo 3)

In [ ]:
from spectra import extract_all_spectra, export_msp

spectra = extract_all_spectra(data, peaks)
print(f"Espectros extraidos: {len(spectra)}")
print(f"Con posible co-elucion: {sum(1 for s in spectra if s.is_coeluted)}")

# Exportar a .msp (necesario para que nistms$.exe los lea)
msp_path = config.OUTPUT_DIR / f"{data.name}_spectra.msp"
export_msp(spectra, msp_path, sample_name=data.name)
print(f"Exportados a: {msp_path}")

In [ ]:
from visualization import plot_spectrum

# Espectro del pico mas intenso
top_peak = peaks.table.nlargest(1, 'intensity').iloc[0]
sp = spectra[int(top_peak['peak_id']) - 1]

fig = plot_spectrum(
    sp.mz, sp.intensities,
    title=f"Espectro de masas -- Pico {sp.peak_id} (RT={sp.rt_min:.2f} min)"
)
plt.show()

print(f"\nFragmentos principales (m/z : intensidad):")
for idx in sp.intensities.argsort()[-10:][::-1]:
    print(f"  m/z {sp.mz[idx]:6.1f} : {sp.intensities[idx]:10.0f}")

---
## 5. Identificacion de compuestos (Modulo 4)

Carga la libreria NIST exportada (.msp) y compara cada espectro usando similitud coseno.  
Devuelve los top 3 candidatos por pico con score >= 0.7.

In [ ]:
from nist_search import load_nist_library, match_all_spectra, build_identification_table

# Cargar libreria .msp exportada de NIST
library = load_nist_library()

if not library:
    print("ERROR: No se pudo cargar la libreria .msp.")
    print(f"  Ruta configurada: {config.NIST_MSP_PATH}")
    print()
    print("Pasos para exportar MAINLIB desde NIST MS Search:")
    print("  1. Abre NIST MS Search (nistms.exe)")
    print("  2. Ve a Tools -> Lib. -> NIST to MSP (o Library Export)")
    print("  3. Selecciona MAINLIB")
    print("  4. Guardalo como:", config.NIST_MSP_PATH)
    print("  5. Vuelve a ejecutar esta celda")
else:
    print(f"Libreria cargada: {len(library)} espectros de referencia")
    print(f"Espectros a buscar: {len(spectra)}")
    print(f"Umbral coseno: {config.COSINE_THRESHOLD}")
    print(f"Top candidatos por pico: {config.NIST_TOP_N}")

In [ ]:
# Matching contra libreria .msp usando similitud coseno
matches = match_all_spectra(
    spectra, library,
    top_n=config.NIST_TOP_N,
    threshold=config.COSINE_THRESHOLD,
    sample_name=data.name,
)
id_table = build_identification_table(spectra, matches)

# Contar picos identificados
n_peaks_total = len(spectra)
n_identified = id_table.loc[id_table['rank'] == 1, 'compound_name'].ne('No match').sum()
print(f"\nPicos identificados: {n_identified} / {n_peaks_total}")

# Guardar tabla completa
id_csv = config.OUTPUT_DIR / f"{data.name}_identifications.csv"
id_table.to_csv(id_csv, index=False)
print(f"Tabla guardada en: {id_csv}")

# --- Mostrar tabla resumen: top 3 candidatos por pico ---
print(f"\n{'='*100}")
print(f"IDENTIFICACIONES -- Top {config.NIST_TOP_N} candidatos por pico (coseno >= {config.COSINE_THRESHOLD})")
print(f"{'='*100}")

cols_display = ['peak_id', 'rt_min', 'rank', 'compound_name', 'formula', 'cas',
                'match_score', 'n_matched_peaks']
identified = id_table[id_table['compound_name'] != 'No match'][cols_display]

if len(identified) > 0:
    display(identified.reset_index(drop=True))
else:
    print(f"\nNingun pico supero el umbral de similitud coseno ({config.COSINE_THRESHOLD})")
    print("Puedes bajar COSINE_THRESHOLD en config.py si es necesario.")

---
## 6. Resumen y archivos exportados

In [ ]:
from visualization import plot_match_score_histogram

# Guardar plots
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
plot_tic_preprocessing(data, save_path=config.OUTPUT_DIR / f"{data.name}_preprocessing.png")
plot_peaks(data, peaks, save_path=config.OUTPUT_DIR / f"{data.name}_peaks.png")
peaks.table.to_csv(config.OUTPUT_DIR / f"{data.name}_peaks.csv", index=False)

# Histograma de match scores (solo rank 1 = mejor candidato)
best_matches = id_table[id_table['rank'] == 1].copy()
if n_identified > 0:
    fig = plot_match_score_histogram(best_matches)
    plt.show()

print(f"\n{'='*60}")
print(f"RESUMEN: {data.name}")
print(f"{'='*60}")
print(f"  Picos detectados: {peaks.n_peaks}")
print(f"  Picos identificados (coseno >= {config.COSINE_THRESHOLD}): {n_identified}")
if n_identified > 0:
    pct = n_identified / peaks.n_peaks * 100
    print(f"  Porcentaje de identificacion: {pct:.1f}%")
print(f"\nArchivos en: {config.OUTPUT_DIR}")
for f in sorted(config.OUTPUT_DIR.glob(f"{data.name}*")):
    print(f"  {f.name}")

---
## Ajuste de parametros

Para procesar otra muestra, vuelve a la seccion 1 y cambia `FILE_IDX`.

Si necesitas ajustar la sensibilidad, edita `config.py`:

| Parametro | Efecto al aumentar |
|---|---|
| `PEAK_MIN_PROMINENCE` | Menos picos (solo los mas intensos) |
| `PEAK_MIN_DISTANCE` | Menos picos cercanos entre si |
| `PEAK_SNR_THRESHOLD` | Elimina picos con mas ruido |
| `SAVGOL_WINDOW` | Mas suavizado (puede perder picos estrechos) |
| `BASELINE_LAMBDA` | Baseline mas suave / rigida |
| `APEX_SCANS_AVERAGE` | Mas scans promediados (mas robusto, menos resolucion) |
| `NIST_MATCH_THRESHOLD` | Menos identificaciones pero mas fiables |